# Deliverable 2 - INF201 

Submission deadline: 30th Oct 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit the Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebooks
    * Should contain code, results/plots and discussion in markdown cells
* Also submit the output files generated by your code
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together

## Scenario  - Campus Coffee Shop Sales Pipeline
You have been hired as a Python developer for the Campus Coffee Shop Chain. The chain operates several coffee stands across different campus buildings. This is an international campus, so the stands accept payment in several currencies.

Each cash register saves transaction logs as text files. However, the legacy system has several issues:
* Mixed Character Encodings: Older registers write text files using ISO-8859-1 (Latin-1) encoding, while newer registers write files using UTF-8.
* System Noise: Registers intersperse valid sales lines with printer and network debug log lines.
* Nested Folder Hierarchy: Log files are stored across nested subdirectories organized by year, store, and register ID.
* Data Errors: System glitches record occasional extreme price outliers (e.g., $999.00), and prices in NOK are by error registered using EUR-scale values (e.g., 4.50 NOK instead of 45.00 NOK).

Your task is to set up Python code that discovers all log files, cleans the transaction data, calculates price metrics and generates summary reports.<br>
Note: In this assignment, you write the functions you will use in tasks 1-4, and then put everything together in task 5. You should therefore read through the entire assignment text before you start, to fully understand what you will use each function for.

### Overview of inputs and outputs for this assignment
Inputs (in `data/` folder)
* You generate the input data on you own laptop by running the script `generate_data_for_d2.py`
* `data/store_config.json`: A nested JSON file containing store metadata.
* `data/2026/...`: A folder hierarchy containing subdirectories with .log files in mixed character encodings.

Outputs (files your code should generate)
* `cleaned_transactions.csv`: A CSV table containing all parsed, outlier-cleaned, and currency-corrected transactions.
* `hourly_revenue.json`: A JSON report containing total revenue per hour per item.
* `status_pivot_table.csv`: A CSV file containing a pivot table of total transaction revenue grouped by item across status categories.
* `item_refund_ratios.csv`: A CSV table containing refund ratios for each item type.

## Task 1: File ingestion and regular expressions

1. Write a robust file reader function:
    * Use `pathlib.Path` to handle file paths.
    * The function should be able to handle both `UTF-8` and `ISO-8859-1/Latin-1` encoding. You may assume that these are the only two encodings the files can have. Note that the function should handle the encoding, you may not hard-code which encoding each file has.
    * Return a tuple containing two strings: `(file_text_content, encoding_used)`.
2. Write a log parser function that uses regular expressions.
    Log files contain valid sales lines mixed with system noise.<br>
    Valid line format:<br>
    `[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)`<br>
    Noise lines to ignore:<br>
    `--- PRINTER PAPER LOW WARNING ---`<br>
    `=== NETWORK SYNC OK ===`<br>
    * Construct a regular expression (`re` module) to capture the 6 key fields from valid lines:
        * `timestamp` (string, format: `YYYY-MM-DD HH:MM:SS`)
        * `register_id` (string, e.g., `POS_01`)
        * `item` (string, e.g., `Latte`)
        * `price` (float, e.g., `4.50`)
        * `currency` (string, e.g., `EUR`, `NOK`, `USD`)
        * `status` (string: `SUCCESS`, `REFUND`, or `CANCELLED`)
    * Return a list of Python dictionaries, where each dictionary represents one valid transaction record.

In [19]:
from pathlib import Path
import re

def robust_file_reader(file_path: Path) -> tuple[str, str]:
    """ checks the encoding of a file and returns its content and encoding type."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        return content, "utf-8"
        
    except UnicodeDecodeError:
        with open(file_path, "r", encoding="iso-8859-1") as f:
            content = f.read()
        return content, "iso-8859-1"
        

def log_parser(file_content: str) -> list[dict]:
    """uses regex on the file to return a list of dictionaries"""
    pattern = r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] \[(?P<register_id>\w+)\] item=(?P<item>\w+) price=(?P<price>\d+\.\d{2}) (?P<currency>\w+) \((?P<status>\w+)\)"
    
    transactions = []
    
    for match in re.finditer(pattern, file_content):
        data = match.groupdict()
        
        data['price'] = float(data['price'])
        
        transactions.append(data)
        
    return transactions



## Task 2: Recursive utility functions

1. Write a recursive directory search function that finds all log files in a nested directory and returns a list of `pathlib.Path` objects:
    * Convert the input path to a `pathlib.Path` object.
    * Iterate through items using `Path.iterdir()`.
    * If an item is a directory, recursively search inside it.
    * If an item is a file with a matching extension (`.log`), add its Path object to a list.
    * Constraint: Do NOT use `Path.glob()`, `Path.rglob()`, or `os.walk()`. You must implement explicit recursion.
2. Metadata about the different stores can be found in the `data/store_config.json` file. This file contains deeply nested dictionaries and lists (see example below). Write a recursive function that:
    * Traverses any combination of nested dictionaries and lists found in this file.
    * Returns a set (type hint: `set[str]`) of all register id strings where `"active": true`.

Example config.json:
```python
{
  "company": "Campus Coffee",
  "regions": [
    {
      "region_name": "Main Campus",
      "stores": [
        {
          "name": "Library Cafe",
          "registers": [
            {"id": "POS_01", "active": true},
            {"id": "POS_02", "active": false}
          ]
        }
      ]
    }
  ]
}
```

In [20]:
from pathlib import Path

def find_log_files(directory: Path) -> list[Path]:
    """recursive function taht finds all log files in directory"""
    
    log_files = []
    
    for item in directory.iterdir():
        
        if item.is_file() and item.suffix == ".log":
            log_files.append(item)

        #have to use recursion to search for log files inside subdirectories
        elif item.is_dir():
            log_files.extend(find_log_files(item))
            
    return log_files

import json
from pathlib import Path


def get_active_registers(data: dict | list) -> set[str]:
    """recursively checks a nested dict or list returns a set of all active register IDs"""
    active_regs = set()
    
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_regs.add(data["id"])
        
        for value in data.values():
            active_regs.update(get_active_registers(value))
            
    elif isinstance(data, list):
        for item in data:
            active_regs.update(get_active_registers(item))
            
    return active_regs


## Task 3: Functions for cleaning and analysing data using NumPy
1. Write a function for removing price outliers that:
    * Converts input price data to a 1D NumPy float array.
    * Detects hardware glitch outliers ($999.00$) where a value deviates from the mean by more than 3 standard deviations:$$\lvert x - \mu \rvert > 3\sigma$$
    * Replaces detected outlier values with the median of the valid prices.
    * Returns the cleaned 1D array.
2. Write a function `compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray` that:
    * Computes the element-wise refund ratio using NumPy array operations:
    $$\text{Refund Ratio} = \frac{\text{Refund Count}}{\text{Sales Count} + 10^{-6}}$$
    * The addition of $\epsilon = 10^{-6}$ prevents zero-division errors.

In [21]:
import numpy as np


def remove_price_outliers(prices) -> np.ndarray:
    """replaces outliers (more than 3 standard deviations from the mean) with the median of the valid prices"""
    # 1D float array (np.array makes a copy, so the input is not changed)
    prices = np.array(prices, dtype=float).ravel()

    mean = prices.mean()
    std = prices.std()

    # True for every price where |x - mean| > 3 * std
    is_outlier = np.abs(prices - mean) > 3 * std

    # median of the prices that are NOT outliers (~ means "not")
    median_valid = np.median(prices[~is_outlier])

    # use the median where is_outlier is True, otherwise keep the original price
    return np.where(is_outlier, median_valid, prices)


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """computes refund ratio = refunds / (sales + 1e-6) for each element"""
    epsilon = 1e-6  # avoids division by zero
    return refund_count / (sales_count + epsilon)

Quick tests of the two functions:

In [22]:
test_prices = [4.5, 3.0, 4.2] * 10 + [999.0]
cleaned = remove_price_outliers(test_prices)
print("max before:", max(test_prices), "| max after:", cleaned.max())

print(compute_refund_ratio(np.array([10, 4, 0]), np.array([2, 1, 0])))

max before: 999.0 | max after: 4.5
[0.19999998 0.24999994 0.        ]


## Task 4: Data analysis using Pandas
1. Write a function that:
    * Converts the list of transaction dictionaries read from the log files into a `pandas.DataFrame`.
    * Converts the `timestamp` column to `datetime` and sets it as the DataFrame index.
    * Sorts the DataFrame chronologically by timestamp.
2. Write a function that has the DataFrame you just created as input, groups data by item and resamples transactions in 1-hour windows (1h), calculating total price (sum).
3. Write a function that has the DataFrame as input  and creates a pivot table displaying total transaction revenue grouped by item across status categories (`SUCCESS`, `REFUND`, `CANCELLED`).
4. Write a function that has the DataFrame as input and calculates the refund ratio per item by calling the function you made in task 3.

In [23]:
import pandas as pd

STATUSES = ["SUCCESS", "REFUND", "CANCELLED"]


def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """converts the list of transaction dicts to a DataFrame with a datetime index, sorted by time"""
    df = pd.DataFrame.from_records(transactions)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    return df.sort_index()


def hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """total price per item in 1-hour windows"""
    return df.groupby("item")["price"].resample("1h").sum()


def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """pivot table with total revenue per item (rows) and status (columns)"""
    pivot = df.pivot_table(values="price", index="item", columns="status", aggfunc="sum", fill_value=0)
    return pivot.reindex(columns=STATUSES, fill_value=0)  # always the same 3 columns


def item_refund_ratios(df: pd.DataFrame) -> pd.Series:
    """refund ratio per item, using compute_refund_ratio from task 3"""
    # number of transactions per item and status
    counts = df.groupby(["item", "status"]).size().unstack(fill_value=0)
    counts = counts.reindex(columns=STATUSES, fill_value=0)

    ratios = compute_refund_ratio(counts["SUCCESS"].to_numpy(), counts["REFUND"].to_numpy())
    return pd.Series(ratios, index=counts.index, name="refund_ratio")

Test on a tiny hand-made table (Latte has 2 sales and 1 refund, so its ratio should be about 0.5):

In [24]:
test_transactions = [
    {"timestamp": "2026-10-15 08:10:00", "register_id": "POS_01", "item": "Latte", "price": 4.5, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 07:50:00", "register_id": "POS_01", "item": "Latte", "price": 4.6, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 08:40:00", "register_id": "POS_02", "item": "Latte", "price": 4.4, "currency": "EUR", "status": "REFUND"},
    {"timestamp": "2026-10-15 09:05:00", "register_id": "POS_02", "item": "Muffin", "price": 3.5, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 09:20:00", "register_id": "POS_02", "item": "Muffin", "price": 3.4, "currency": "EUR", "status": "CANCELLED"},
]
test_df = create_dataframe(test_transactions)

print(test_df, "\n")
print(hourly_revenue(test_df), "\n")
print(status_pivot_table(test_df), "\n")
print(item_refund_ratios(test_df))

                    register_id    item  price currency     status
timestamp                                                         
2026-10-15 07:50:00      POS_01   Latte    4.6      EUR    SUCCESS
2026-10-15 08:10:00      POS_01   Latte    4.5      EUR    SUCCESS
2026-10-15 08:40:00      POS_02   Latte    4.4      EUR     REFUND
2026-10-15 09:05:00      POS_02  Muffin    3.5      EUR    SUCCESS
2026-10-15 09:20:00      POS_02  Muffin    3.4      EUR  CANCELLED 

item    timestamp          
Latte   2026-10-15 07:00:00    4.6
        2026-10-15 08:00:00    8.9
Muffin  2026-10-15 09:00:00    6.9
Name: price, dtype: float64 

status  SUCCESS  REFUND  CANCELLED
item                              
Latte       9.1     4.4        0.0
Muffin      3.5     0.0        3.4 

item
Latte     0.5
Muffin    0.0
Name: refund_ratio, dtype: float64


## Task 5: Create full workflow
Combine the functions you have made into a workflow that does the following:
1. Recursively find all log files and extract active register IDs.
2. Ingest text logs and parse records into a list of dictionaries.
3. Build the initial Pandas DataFrame.
4. Apply NumPy outlier cleaning to remove $999.00$ price glitches.
5. Apply currency correction: multiply price by $10.0$ for all transactions where `currency == "NOK"`.
6. Generate resampled reports, pivot tables and refund ratios.
7. Save all output files listed in the introduction (Scenario: Overview of inputs and outputs for this assignment).

In [25]:
DATA_DIR = Path("d2_data")  # folder created by generate_data_for_d2.py


def run_workflow(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """runs the full workflow: find logs, parse, clean, analyse and save the 4 output files"""
    # 1. find all log files and the active registers
    log_files = find_log_files(data_dir)
    with open(data_dir / "store_config.json", encoding="utf-8") as f:
        active_registers = get_active_registers(json.load(f))
    print(f"1. Found {len(log_files)} log files. Active registers: {sorted(active_registers)}")

    # 2. read the logs and parse them to a list of dictionaries
    transactions = []
    for file_path in sorted(log_files):
        content, encoding = robust_file_reader(file_path)
        file_transactions = log_parser(content)
        transactions.extend(file_transactions)
        print(f"   {file_path.relative_to(data_dir)}: {len(file_transactions)} transactions [{encoding}]")

    ## keep only transactions from active registers
    transactions = [t for t in transactions if t["register_id"] in active_registers]
    print(f"2. {len(transactions)} transactions from active registers")

    # 3. build the DataFrame
    df = create_dataframe(transactions)
    print(f"3. DataFrame with {len(df)} rows")

    # 4. remove the 999.00 price glitches with numpy
    glitches_before = (df["price"] == 999.0).sum()
    df["price"] = remove_price_outliers(df["price"])
    glitches_after = (df["price"] == 999.0).sum()
    print(f"4. Prices of 999.00: {glitches_before} before cleaning, {glitches_after} after")

    # 5. currency correction: NOK prices were registered on a EUR scale, so multiply by 10
    is_nok = df["currency"] == "NOK"
    df.loc[is_nok, "price"] = df.loc[is_nok, "price"] * 10.0
    print(f"5. Multiplied {is_nok.sum()} NOK prices by 10")

    # 6. reports
    hourly = hourly_revenue(df)
    pivot = status_pivot_table(df)
    ratios = item_refund_ratios(df)
    print("6. Reports created")

    # 7. saving the output file
    df.to_csv("cleaned_transactions.csv", float_format="%.2f")

    hourly_report = {}  # {item: {hour: total}}
    for (item, hour), total in hourly.items():
        hourly_report.setdefault(item, {})[hour.strftime("%Y-%m-%d %H:%M")] = round(float(total), 2)
    with open("hourly_revenue.json", "w", encoding="utf-8") as f:
        json.dump(hourly_report, f, indent=2)

    pivot.to_csv("status_pivot_table.csv", float_format="%.2f")
    ratios.to_csv("item_refund_ratios.csv", float_format="%.4f")
    print("7. Saved cleaned_transactions.csv, hourly_revenue.json, status_pivot_table.csv, item_refund_ratios.csv")

    return df

In [26]:
cleaned_df = run_workflow()

1. Found 9 log files. Active registers: ['POS_01', 'POS_02', 'POS_03']
   2026\library_cafe\POS_01\register_sales.log: 80 transactions [utf-8]
   2026\library_cafe\POS_02\register_sales.log: 80 transactions [utf-8]
   2026\library_cafe\POS_03\register_sales.log: 80 transactions [utf-8]
   2026\science_building\POS_01\register_sales.log: 80 transactions [utf-8]
   2026\science_building\POS_02\register_sales.log: 80 transactions [utf-8]
   2026\science_building\POS_03\register_sales.log: 80 transactions [utf-8]
   2026\student_union\POS_01\register_sales.log: 80 transactions [utf-8]
   2026\student_union\POS_02\register_sales.log: 80 transactions [utf-8]
   2026\student_union\POS_03\register_sales.log: 80 transactions [utf-8]
2. 720 transactions from active registers
3. DataFrame with 720 rows
4. Prices of 999.00: 32 before cleaning, 0 after
5. Multiplied 247 NOK prices by 10
6. Reports created
7. Saved cleaned_transactions.csv, hourly_revenue.json, status_pivot_table.csv, item_refund_ra

Look at the results:

In [27]:
print(cleaned_df.head(), "\n")
print("Highest price after cleaning:", cleaned_df["price"].max(), "\n")
print(pd.read_csv("status_pivot_table.csv"), "\n")
print(pd.read_csv("item_refund_ratios.csv"))

                    register_id        item  price currency   status
timestamp                                                           
2026-10-15 07:00:12      POS_02      Muffin  3.270      USD  SUCCESS
2026-10-15 07:00:50      POS_02    Sandwich  7.340      EUR  SUCCESS
2026-10-15 07:00:56      POS_03  Cappuccino  4.055      USD  SUCCESS
2026-10-15 07:01:02      POS_01       Latte  4.080      EUR  SUCCESS
2026-10-15 07:01:09      POS_03      Muffin  3.050      EUR  SUCCESS 

Highest price after cleaning: 80.0 

         item  SUCCESS  REFUND  CANCELLED
0  Cappuccino  1505.76  537.30     648.73
1   Croissant  1126.62  172.68     368.34
2    Espresso   760.71  287.86     248.80
3       Latte  1295.47  140.31     428.22
4      Muffin  1110.30  219.00     410.49
5    Sandwich  2746.66  751.32     454.51 

         item  refund_ratio
0  Cappuccino        0.3494
1   Croissant        0.1389
2    Espresso        0.3194
3       Latte        0.1728
4      Muffin        0.2111
5    Sandwich 

## Repository and commit log
* GitHub repository: https://github.com/sultanajafrin/Inf_201_deliverable_02
* Commit log: `git_log.txt`
* Virtual environment: `requirements.txt` is in the repository

## Discussion

### 1. What we did
We wrote Python code that reads the cash register log files, cleans the data and makes four reports. The code finds 9 log files in nested folders, and the 9 files contain 720 valid sales lines. The other lines (printer, network and debug messages) are noise and are ignored.

### 2. How the main parts work
* **Reading files:** The reader tries UTF-8 first. If that fails, it reads the file as ISO-8859-1. We do not choose the encoding by hand. In our data all 9 files were read as UTF-8, because the files only contain plain English letters, which look the same in both encodings.
* **Regular expression:** One pattern picks out the 6 fields (time, register, item, price, currency, status) from each sales line. Noise lines do not match the pattern, so they are skipped.
* **Recursion:** One function calls itself to find .log files in folders inside folders. Another calls itself to look through the dictionaries and lists in the JSON file. It found 3 active registers: POS_01, POS_02 and POS_03.
* **NumPy cleaning:** A price is an outlier if it is more than 3 standard deviations from the mean. We replace outliers with the median of the other prices.
* **Pandas reports:** The data goes into a table with the time as index. Then we make revenue per hour for each item, a pivot table of item against status, and the refund ratio for each item.

### 3. Results
* The data had 32 prices of 999.00 (the hardware glitch). After the cleaning there were none left.
* 247 of the 720 transactions were in NOK. Their prices were 10 times too small, so we multiplied them by 10. We did this after removing the outliers.
* The highest price after cleaning is 80.00 (a sandwich paid in NOK).
* Sandwich has the highest revenue from successful sales (2746.66). Espresso has the lowest (760.71).
* The refund ratio is the number of refunds divided by the number of successful sales. Cappuccino has the highest ratio (0.3494) and Croissant the lowest (0.1389). The ratios for the other items are Espresso 0.3194, Sandwich 0.2247, Muffin 0.2111 and Latte 0.1728.
* The statuses in the generated data are chosen at random, so the differences between items are most likely chance and not a real pattern.

### 4. Problems and limits
* The 3-sigma rule uses the mean and the standard deviation, and the outliers themselves make both larger. With only a few glitches this works well, but with many outliers some of them could be missed.
* EUR, USD and NOK are not converted into one currency. The revenue totals therefore add numbers in different currencies together. A real system should convert everything to one currency first.
* We removed outliers from all prices together. Prices in different currencies have different scales, so cleaning per currency could be more precise.
* We keep only transactions from active registers. In our data all three registers are active, so no transactions were removed.
